# DNS analysis using CHAOS records
Questions:
* How common/reliable are version.bind records for identifying running DNS software
* Which DNS software is often used by DNS anycast operators?
* Do we see different DNS software used? (e.g., PoPs running different software/versions)
Data description:
* Data collected using 245 Ark VPs that support DNS probing
* Targets: 3,363 IPv4 anycast addresses that reply to DNS; 3,240 IPv6 anycast addresses
* Data format (.csv.gz)
* "vp_name" - name of the Ark VP that performed the measurement (includes city and country codes)
* "dst" - probed target address
* "qname" - CHAOS record queried (id.server, hostname.bind, version.bind)
* "rtt_ms" - time difference between sending and receiving (ms)
* "record" - record contained in the DNS response

In [1]:
import pandas as pd

In [2]:
# load in chaos data

chaosv4 = pd.read_csv('2026-01-16_DNSv4.csv.gz')
chaosv6 = pd.read_csv('2026-01-16_DNSv6.csv.gz')

# print(chaosv4.head())

# we only consider version.bind
# chaosv6 = chaosv6[chaosv6['qname'] == 'version.bind']
# chaosv4 = chaosv4[chaosv4['qname'] == 'version.bind']

chaosv4.head()

,vp_name,dst,qname,rtt_ms,record
0,lax6-us,98.158.111.2,id.server,1.04,la-blue.ns-node.iptp.net
1,lax9-us,98.158.111.2,id.server,2.00,la-blue.ns-node.iptp.net
2,lax5-us,161.232.17.29,id.server,1.35,LAX3
3,ful-us,161.232.17.29,id.server,2.17,LAX4
4,bur-us,161.232.17.29,id.server,1.76,LAX3


In [ ]:
# TODO get infra:ns from openintel to see which tld it hosts

In [3]:
# exploratory analysis
print("---ipv4---")
print(f"data from {chaosv4['vp_name'].nunique()} VPs")
print(f"towards {chaosv4['dst'].nunique()} targets that have a record")

# print number of servers supporting each chaos record
for record in chaosv4['qname'].unique():
    f = chaosv4.loc[chaosv4['qname'] == record]
    print(f"{f['dst'].nunique():,} targets that support {record}")



---ipv4---
data from 242 VPs
towards 2169 targets that have a record
1,438 targets that support id.server
2,108 targets that support hostname.bind
1,555 targets that support version.bind
424 targets that support version.server
350 targets that support authors.bind
29 targets that support server.bind


In [4]:
# exploratory analysis
print("---ipv6---")
print(f"data from {chaosv6['vp_name'].nunique()} VPs")
print(f"towards {chaosv6['dst'].nunique()} targets that have a record")

# print number of servers supporting each chaos record
for record in chaosv6['qname'].unique():
    f = chaosv6.loc[chaosv6['qname'] == record]
    print(f"{f['dst'].nunique():,} targets that support {record}")



---ipv6---
data from 73 VPs
towards 2018 targets that have a record
538 targets that support authors.bind
1,824 targets that support hostname.bind
1,133 targets that support id.server
1,584 targets that support version.bind
318 targets that support version.server
23 targets that support server.bind


In [5]:
# which dns software versions are there?

# common record values
chaosv4['record'].value_counts().reset_index()


,record,count
0,-,71041
1,9.11.22,54594
2,087a32084726a0bc478d318e0588d12ebe87fc56,41437
3,CIRA Server,28656
4,UltraDNS Nameserver,15826
...,...,...
25815,960m95,1
25816,1232m111,1
25817,680m21,1
25818,786m94,1


In [6]:
chaosv4[chaosv4['record'] == '9.11.22'].head() # -> bind version used in standard linux


,vp_name,dst,qname,rtt_ms,record
838151,sjc4-us,199.253.61.1,version.bind,1.60,9.11.22
838154,sjc6-us,199.253.61.1,version.bind,0.82,9.11.22
838192,mce-us,199.253.61.1,version.bind,10.61,9.11.22
838227,cld4-us,199.253.61.1,version.bind,19.48,9.11.22
838247,sdm-us,65.22.93.1,version.bind,6.72,9.11.22


In [7]:
chaosv4[chaosv4['record'] == '087a32084726a0bc478d318e0588d12ebe87fc56'].head() #


,vp_name,dst,qname,rtt_ms,record
838232,sdm-us,157.53.235.65,version.bind,36.03,087a32084726a0bc478d318e0588d12ebe87fc56
838246,sdm-us,45.54.97.129,version.bind,14.79,087a32084726a0bc478d318e0588d12ebe87fc56
838282,lax6-us,157.53.235.65,version.bind,33.41,087a32084726a0bc478d318e0588d12ebe87fc56
838284,lax4-us,157.53.235.65,version.bind,34.06,087a32084726a0bc478d318e0588d12ebe87fc56
838307,tij-mx,157.53.235.65,version.bind,33.91,087a32084726a0bc478d318e0588d12ebe87fc56


In [26]:
# init software

chaosv4['software'] = 'unknown'


## powerdns

In [27]:
# different types of powerdns values
chaosv4.loc[
    chaosv4['record'].str.lower().str.contains('powerdns', na=False),
    'software'
] = 'powerdns'

chaosv4[chaosv4['software'] == 'powerdns']['record'].unique()

array(['powerdns-2', 'powerdns-0', 'powerdns-1',
       'slave-gslb-powerdns-g7f2c', 'slave-gslb-powerdns-tz7cz',
       'slave-gslb-powerdns-gk7r9', 'slave-gslb-powerdns-qmmhp',
       'slave-gslb-powerdns-sjqxz', 'slave-gslb-powerdns-v78t2',
       'slave-gslb-powerdns-np4ct', 'slave-gslb-powerdns-qvf4k',
       'slave-gslb-powerdns-qdqbd', 'slave-gslb-powerdns-f5thw',
       'slave-gslb-powerdns-6dfhl', 'slave-gslb-powerdns-55k9v',
       'slave-gslb-powerdns-wwgq7', 'slave-gslb-powerdns-9rnf6',
       'slave-gslb-powerdns-2hvr7', 'slave-gslb-powerdns-mp7cp',
       'slave-gslb-powerdns-nvf8q', 'slave-gslb-powerdns-ntfzq',
       'slave-gslb-powerdns-xxckp', 'slave-gslb-powerdns-dmx5d',
       'slave-gslb-powerdns-z7gkh', 'slave-gslb-powerdns-pvj59',
       'slave-gslb-powerdns-9p54f', 'slave-gslb-powerdns-pj9lm',
       'slave-gslb-powerdns-7wt69', 'slave-gslb-powerdns-5vtwr',
       'slave-gslb-powerdns-bct8z', 'slave-gslb-powerdns-zzt9r',
       'slave-gslb-powerdns-4wjpc', 'slav

In [30]:
# number of dns servers running powerdns
chaosv4[chaosv4['software'] == 'powerdns']['dst'].nunique()

199

## knot

In [32]:
chaosv4.loc[
    chaosv4['record'].str.lower().str.contains('knot', na=False),
    'software'
] = 'knot'

chaosv4[chaosv4['software'] == 'knot']['record'].unique()


array(['reg-ffm1_customer15_knot', 'reg-ffm1_customer14_knot',
       'reg-rey1_customer14_knot', 'reg-prs1_shared1_knot',
       'reg-rey1_shared1_knot', 'ns-1.es.us4.knot', 'ns-2.es.us3.knot',
       'ns-2.es.us4.knot', 'ns-1.es.at1.knot', 'ns-2.es.kr1.knot',
       'ns-1.es.kr1.knot', 'ns-2.cira.us3.knot', 'ns-2.cira.kr1.knot',
       'ns-2.cira.de2.knot', 'ns-1.cira.at1.knot', 'ns-1.cira.de2.knot',
       'reg-ffm1_customer5_knot', 'ns-2.de.kr1.knot',
       'Knot - NIC Chile - cph', 'Knot - NIC Chile - elbosque',
       'ns-1.de.kr1.knot', 'ns-2.lu.us4.knot', 'ns-1.lu.us4.knot',
       'ns-2.lu.kr1.knot', 'ns-1.lu.kr1.knot', 'ns-1.lu.at1.knot',
       'ns-2.fi.us4.knot', 'ns-1.fi.us4.knot', 'ns-1.fi.at1.knot',
       'ns-1.fi.kr1.knot', 'ns-2.fi.kr1.knot',
       'Knot - NIC Chile - amsterdam', 'Knot - NIC Chile - saopaulo',
       'ns-1.eu.us4.knot', 'ns-2.eu.us4.knot', 'ns-1.eu.at1.knot',
       'ns-1.eu.kr1.knot', 'ns-2.eu.kr1.knot',
       'Knot - NIC Chile - londres', 'Knot -

In [33]:
chaosv4[chaosv4['software'] == 'knot']['dst'].nunique()


48

## bind

In [90]:
chaosv4.loc[
    chaosv4['record'].str.lower().str.contains(r'bind|9\.11\.22|michael graff|brian wellington|mark andrews|bob halley|james brister|michael sawyer', case=False, na=False, regex=True),
    'software'
] = 'bind'
chaosv4[chaosv4['software'] == 'bind']['record'].unique()

array(['ns-2.es.nl1.bind', 'ns-1.es.de2.bind', 'ns-1.es.se1.bind',
       'ns-1.es.nl1.bind', 'ns-2.es.se1.bind', 'ns-2.es.de2.bind',
       'ns.es.hk2.bind', 'ns-1.es.de8.bind', 'ns-2.es.br1.bind',
       'ns.es.fr2.bind', 'ns.es.bg1.bind', 'ns.es.in1.bind',
       'ns.es.sg1.bind', 'ns.es.pl1.bind', 'ns-1.es.br1.bind',
       'ns-1.cira.us4.bind', 'ns-2.cira.us4.bind', 'ns-1.cira.de8.bind',
       'ns.cira.bg1.bind', 'ns.cira.pl1.bind', 'ns.cira.fr3.bind',
       'ns.cira.anx-jnb1.bind', 'ns.cira.anx-sydney1.bind',
       'ns.cira.hk2.bind', 'ns-2.cira.se1.bind', 'ns-1.cira.nl1.bind',
       'ns-2.cira.cn1.bind', 'ns-2.cira.nl1.bind', 'ns.cira.anx-uk2.bind',
       'ns.cira.sg1.bind', 'ns-2.cira.br1.bind', 'ns-1.cira.se1.bind',
       'ns.cira.in1.bind', 'ns.cira.fr2.bind', 'ns-1.cira.cn1.bind',
       'ns-1.cira.br1.bind', 'ns-1.de.us4.bind', 'ns.de.mx1.bind',
       'ns-2.de.us3.bind', 'ns-2.de.us4.bind', 'ns-1.de.de2.bind',
       'ns.de.sg1.bind', 'ns-2.de.de2.bind', 'ns.de.fr2.b

In [85]:
chaosv4[chaosv4['software'] == 'bind']['dst'].nunique()

356

## redhat

In [36]:
chaosv4.loc[
    chaosv4['record'].str.lower().str.contains('redhat', na=False),
    'software'
] = 'redhat'

chaosv4[chaosv4['software'] == 'redhat']['record'].unique()


array(['9.11.26-RedHat-9.11.26-6.el8',
       '9.3.6-P1-RedHat-9.3.6-4.P1.el5_5.3',
       '9.9.4-RedHat-9.9.4-73.el7_6',
       '9.11.4-P2-RedHat-9.11.4-16.P2.el7_8.6',
       '9.11.4-P2-RedHat-9.11.4-26.P2.el7_9.13',
       '9.11.4-P2-RedHat-9.11.4-9.P2.el7',
       '9.11.4-P2-RedHat-9.11.4-26.P2.el7_9.4',
       '9.9.4-RedHat-9.9.4-61.el7', '9.11.13-RedHat-9.11.13-6.el8_2.1',
       '9.11.4-P2-RedHat-9.11.4-26.P2.amzn2.13',
       '9.11.4-P2-RedHat-9.11.4-26.P2.amzn2.13.11',
       '9.11.4-P2-RedHat-9.11.4-9.P2.amzn2.0.2',
       '9.11.4-P2-RedHat-9.11.4-9.P2.amzn2.0.4',
       '9.11.4-P2-RedHat-9.11.4-26.P2.amzn2.4',
       '9.11.4-P2-RedHat-9.11.4-26.P2.amzn2.5.2',
       '9.11.36-RedHat-9.11.36-16.el8_10.6',
       '9.11.4-P2-RedHat-9.11.4-26.P2.el7_9.15',
       '9.8.2rc1-RedHat-9.8.2-0.47.rc1.el6',
       '9.11.4-P2-RedHat-9.11.4-26.P2.el7_9.16',
       '9.11.4-P2-RedHat-9.11.4-26.P2.el7_9.10'], dtype=object)

In [37]:
chaosv4[chaosv4['software'] == 'redhat']['dst'].nunique()

23

## unbound

In [41]:
chaosv4.loc[
    chaosv4['record'].str.lower().str.contains('unbound', na=False),
    'software'
] = 'unbound'

chaosv4[chaosv4['software'] == 'unbound']['record'].unique()


array([], dtype=object)

In [42]:
chaosv4[chaosv4['software'] == 'unbound']['dst'].nunique()


0

## core

In [43]:
chaosv4.loc[
    chaosv4['record'].str.lower().str.contains('core', na=False),
    'software'
] = 'core'

chaosv4[chaosv4['software'] == 'core']['record'].unique()


array(['drc-v-wgdns-3.be.core.pw', 'drc-v-wgdns-4.be.core.pw',
       'anx-v-wgdns-3.be.core.pw', 'anx-v-wgdns-4.be.core.pw',
       'am3-v-wgdns-3.be.core.pw', 'am3-v-wgdns-4.be.core.pw',
       'sg2-v-wgdns-3.be.core.pw', 'sg2-v-wgdns-4.be.core.pw',
       'core-ns-edge-uslax-0', 'core-ns-edge-uslax-1',
       'core-ns-edge-usnyc-1', 'core-ns-edge-usnyc-0',
       'core-ns-edge-sgsin-0', 'core-ns-edge-esmad-0',
       'core-ns-edge-defra-0', 'core-ns-edge-frpar-0',
       'core-ns-edge-nlams-0', 'core-ns-edge-uklon-0',
       'core-ns-edge-sgsin-1', 'core-ns-edge-sesto-0',
       'core-ns-edge-itmil-0', 'core-ns-edge-lubet-1',
       'core-ns-edge-lubet-0', 'core-ns-edge-lukyl-0',
       'core-ns-edge-hubud-0', 'core01a.adectra.com',
       'dnsaut-co-02-parild2.core.wifirst.net',
       'dnsaut-co-02-loneq8.core.wifirst.net',
       'dnsaut-co-02-lonthe.core.wifirst.net',
       'dnsaut-co-02-pareq2.core.wifirst.net', 'CoreDNS-3.0.0',
       'CoreDNS-3.0.1', 'CoreDNS-2.9.13'], dtype

In [44]:
chaosv4[chaosv4['software'] == 'core']['dst'].nunique()

9

## dnsmasq

In [73]:
chaosv4.loc[
    chaosv4['record']
        .str.lower()
        .str.contains(r'dnsmasq|simon kelley', na=False),
    'software'
] = 'dnsmasq'

chaosv4[chaosv4['software'] == 'dnsmasq']['record'].unique()


array(['dnsmasq-2.83', 'dnsmasq-2.78', 'Simon Kelley'], dtype=object)

In [74]:
chaosv4[chaosv4['software'] == 'dnsmasq']['dst'].nunique()


1555

# ultradns (propietary)

In [94]:
chaosv4.loc[
    chaosv4['record']
        .str.lower()
        .str.contains(r'ultradns|ultra', na=False),
    'software'
] = 'ultradns'

chaosv4[chaosv4['software'] == 'ultradns']['record'].unique()


array(['rcrsv1.uslax1.ultradns', 'rcrsv2.uslax1.ultradns',
       'rcrsv4.uslax1.ultradns', 'rcrsv1.ussea1.ultradns',
       'rcrsv3.uslax1.ultradns', 'rcrsv2.usdal1.ultradns',
       'rcrsv4.usdal1.ultradns', 'rcrsv2.cator1.ultradns',
       'rcrsv4.jptyo1.ultradns', 'rcrsv1.usdal1.ultradns',
       'rcrsv3.uswas1.ultradns', 'rcrsv2.uswas1.ultradns',
       'rcrsv1.uswas1.ultradns', 'rcrsv4.uswas1.ultradns',
       'rcrsv3.cator1.ultradns', 'rcrsv2.jptyo1.ultradns',
       'rcrsv4.cator1.ultradns', 'rcrsv4.usnyc1.ultradns',
       'rcrsv1.usmia1.ultradns', 'rcrsv1.usatl1.ultradns',
       'rcrsv1.uschi1.ultradns', 'rcrsv4.gblon1.ultradns',
       'rcrsv2.sesto1.ultradns', 'rcrsv3.sesto1.ultradns',
       'rcrsv1.sesto1.ultradns', 'rcrsv1.defra1.ultradns',
       'rcrsv3.gblon1.ultradns', 'rcrsv3.defra1.ultradns',
       'rcrsv3.usdal1.ultradns', 'rcrsv2.uschi1.ultradns',
       'rcrsv2.defra1.ultradns', 'rcrsv4.ausyd1.ultradns',
       'rcrsv2.gblon1.ultradns', 'rcrsv2.brsao1.ultradns

In [95]:
chaosv4[chaosv4['software'] == 'ultradns']['dst'].nunique()

91

# analysis

In [109]:
# print vps that see dnsmasq the msot often

dnsmasq_data = chaosv4[chaosv4['software'] == 'dnsmasq']

# Count unique destinations per VP
vp_summary = dnsmasq_data.groupby('vp_name').agg(
    unique_dst=('dst', 'nunique')
).reset_index()

# Sort descending to see the VPs that see the most destinations
vp_summary = vp_summary.sort_values(by='unique_dst', ascending=False)

vp_summary

,vp_name,unique_dst
0,blq-it,1555
1,bwi3-us,1555


In [111]:
# exclude blq-it and bwi3-us -> hitting local dnsmasq resolvers

chaosv4 = chaosv4[chaosv4['vp_name'] != 'blq-it']
chaosv4 = chaosv4[chaosv4['vp_name'] != 'bwi3-us']


In [112]:
summary = chaosv4.groupby('software').agg(
    unique_dst=('dst', 'nunique'),
    unique_qname=('qname', 'nunique'),
    unique_record=('record', 'nunique')
).reset_index()

# Aggregate all "known" software (software != 'unknown')
known_row = {
    'software': 'known',
    'unique_dst': chaosv4.loc[chaosv4['software'] != 'unknown', 'dst'].nunique(),
    'unique_qname': chaosv4.loc[chaosv4['software'] != 'unknown', 'qname'].nunique(),
    'unique_record': chaosv4.loc[chaosv4['software'] != 'unknown', 'record'].nunique()
}

# Aggregate total (all software)
total_row = {
    'software': 'total',
    'unique_dst': chaosv4['dst'].nunique(),
    'unique_qname': chaosv4['qname'].nunique(),
    'unique_record': chaosv4['record'].nunique()
}

# Append both rows
summary = pd.concat([summary, pd.DataFrame([known_row, total_row])], ignore_index=True)

summary

,software,unique_dst,unique_qname,unique_record
0,bind,356,5,196
1,core,9,4,33
2,knot,48,4,70
3,powerdns,199,3,249
4,redhat,23,1,20
5,ultradns,91,4,230
6,unknown,2130,6,25018
7,known,682,5,798
8,total,2168,6,25816


In [113]:
unknown = chaosv4[chaosv4['software'] == 'unknown']

unique_dsts_per_record = unknown.groupby('record')['dst'].nunique().reset_index().sort_values(by='dst', ascending=False)
unique_dsts_per_record

,record,dst
267,-,321
9924,Ben Cottrell,314
10223,Damien Neil,314
10365,Jeremy C. Reed,313
10224,Danny Mayer,313
...,...,...
13209,cache-toj-leto2350063,1
13210,cache-toj-leto2350065,1
13211,cache-toj-leto2350068,1
13212,cache-toj-leto2350069,1


In [114]:
known = chaosv4[chaosv4['software'] != 'unknown']

In [115]:
# get software per dst

dst_software_counts = known.groupby('dst')['software'].nunique()

dst_multiple_software = dst_software_counts[dst_software_counts > 1].index

dst_multiple_software_rows = known[known['dst'].isin(dst_multiple_software)]

dst_multiple_software_rows[['dst', 'software']].drop_duplicates()


,dst,software
56376,194.0.33.53,knot
59786,194.0.33.53,bind
74517,45.142.220.101,bind
74584,45.142.220.101,knot
119263,194.246.96.1,bind
...,...,...
1372050,195.114.29.130,bind
1372408,109.205.191.4,bind
1373625,91.237.248.2,bind
1376018,5.188.175.100,bind


In [116]:
# print software found for each

dst_software_list = (
    dst_multiple_software_rows
    .groupby('dst')['software']
    .agg(lambda x: list(x.unique()))
    .reset_index()
)

dst_software_list

,dst,software
0,103.51.60.8,"[redhat, bind]"
1,109.205.191.4,"[redhat, bind]"
2,13.248.177.206,"[redhat, bind]"
3,141.164.24.204,"[redhat, bind]"
4,15.197.207.2,"[core, redhat, bind]"
5,15.197.237.120,"[redhat, bind]"
6,170.247.170.2,"[knot, bind]"
7,170.247.171.2,"[bind, knot]"
8,185.131.147.3,"[redhat, bind]"
9,185.143.80.1,"[knot, bind]"


In [ ]:
# 103.51.60.8 -> mongolian bank
# 109.205.191.4 -> moldovian telecom?

# 193.0.14.129 -> k-root
# 170.247.171.2 -> b-root

# 192.175.48.1 & 192.31.196.1 (AS112)
# 199.244.90.99 (neustar public recursive resolver)


# AWS global accelorator (route 53)
# 13.248.177.206
# 15.197.207.2
# 15.197.237.120
# 170.247.170.2 & 170.247.171.2
# 3.33.202.173 & 3.33.216.193
# 76.223.1.101 & 76.223.53.100

# google dns endpoint
# 35.243.187.176

# edgecast (verizon) tld auth
# 69.31.181.100

# gcore / swiftycdn
# 5.188.175.100 & 91.192.82.100

# cira (tld auth)
# 45.142.220.101

# apnic rdns
# 103.51.60.8

# ripe admin and rdns
# 109.205.191.4, 193.0.8.254, & 193.0.15.254

# lacnic region ISP
# 190.52.162.69

# frontier networks public resolver
# 45.2.2.2

# russian telecom
# 78.37.72.203

# south korean IX DNS provider
# 141.164.24.204

# clouvider
# 185.131.147.3

# epik nameserver
# 185.143.80.1: